# Standardised Data Cleaning Notebook

This notebook **standardises** the **data cleaning** process for **historical stablecoin datasets**. Users should **update** the file paths for both *data loading and saving* accordingly. The cleaned data is saved in **Parquet** format to *preserve data types*. This workflow can be easily replicated across other stablecoin datasets by modifying the file paths.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Cleaning

In [ ]:
# load data
stablecoin = "ust" # or "usdc", "usdt", "pax", "dai", "ust"
path = f"raw_data/{stablecoin}_hist_data.csv"
df = pd.read_csv(path)
# for ust only 
# df = df[df["timestamp"] <= "2022-05-11"].copy()

In [24]:
# parse datetimes + numerics
time_cols = ["timestamp", "timeOpen", "timeClose", "timeHigh", "timeLow"]
for c in time_cols:
    df[c] = pd.to_datetime(df[c])

num_cols = ["open", "high", "low", "close", "volume", "marketCap", "circulatingSupply"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c])

df.dtypes

Unnamed: 0                    int64
symbol                          str
timestamp            datetime64[us]
timeOpen             datetime64[us]
timeClose            datetime64[us]
timeHigh             datetime64[us]
timeLow              datetime64[us]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [26]:
# drop duplicates (same timestamp)
print(f"Number of rows before dropping duplicates: {len(df)}")
df = df.sort_values("timestamp")
df = df.drop_duplicates(subset=["timestamp"], keep="last")
print(f"Number of rows after dropping duplicates: {len(df)}")

Number of rows before dropping duplicates: 532
Number of rows after dropping duplicates: 532


### Basic Price Sanity Checks

In [27]:
# ensure low > 0, high > 0, and low <= high for all observations
# if any violations exist, they should be cleaned before analysis

invalid_low = (df["low"] <= 0).sum()
invalid_high = (df["high"] <= 0).sum()
invalid_order = (df["low"] > df["high"]).sum()

print(invalid_low, invalid_high, invalid_order)

# in this dataset, all values are valid (all counts = 0),
# so no additional cleaning is required for low/high prices.

0 0 0


## Depeg Labelling

In [35]:
# create current depeg label
df["depeg"] = (abs(df["close"] - 1) >= 0.006).astype(int)
# check number of depegs
df["depeg"].sum()

np.int64(33)

In [36]:
df.head(10)

,Unnamed: 0,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply,depeg
0,0,UST,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 11:48:07,2020-11-25 17:19:07,1.010488,1.010752,0.993542,0.999572,3758726.64,1.332804e+07,13333749.41,0
1,1,UST,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 03:33:07,2020-11-26 09:47:07,0.999360,1.022774,0.982691,1.000441,16210369.84,1.564638e+07,15639476.02,0
2,2,UST,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 11:33:07,2020-11-27 16:07:07,1.000106,1.003605,0.994698,0.999515,4086972.72,1.743930e+07,17447758.34,0
3,3,UST,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 19:25:11,2020-11-28 16:04:08,0.999476,1.004033,0.995601,0.999325,3721757.38,1.743599e+07,17447757.88,0
4,4,UST,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 16:06:07,2020-11-29 21:04:07,0.999246,1.001828,0.997059,0.999641,856953.09,1.743870e+07,17444966.35,0
5,5,UST,2020-11-30 23:59:59,2020-11-30,2020-11-30 23:59:59,2020-11-30 15:33:11,2020-11-30 14:49:09,0.999629,1.001512,0.987902,0.997708,2663923.71,1.742875e+07,17468780.46,0
6,6,UST,2020-12-01 23:59:59,2020-12-01,2020-12-01 23:59:59,2020-12-01 12:28:09,2020-12-01 12:29:08,0.996880,1.006454,0.991093,0.999963,9102663.96,1.754190e+07,17542544.13,0
7,7,UST,2020-12-02 23:59:59,2020-12-02,2020-12-02 23:59:59,2020-12-02 09:49:08,2020-12-02 00:19:08,1.000168,1.003055,0.995118,0.999860,2346373.47,2.354619e+07,23549485.24,0
8,8,UST,2020-12-03 23:59:59,2020-12-03,2020-12-03 23:59:59,2020-12-03 05:43:08,2020-12-03 13:22:08,0.999831,1.002302,0.995493,0.999795,3813306.81,2.354382e+07,23548661.59,0
9,9,UST,2020-12-04 23:59:59,2020-12-04,2020-12-04 23:59:59,2020-12-04 10:19:08,2020-12-04 04:09:07,0.999849,1.002200,0.989070,1.000692,4671524.02,2.356496e+07,23548661.59,0


In [37]:
# create future depeg labels (within next h days)
horizons = [1, 3, 5, 7, 14, 30]

for h in horizons:
    df[f"depeg_future_{h}d"] = (
        df["depeg"]
        .shift(-1) # start from tomorrow
        .rolling(window=h, min_periods=1)
        .max()
        .astype("float") # keep NaN at tail first
    )

# handle tail NaNs (no future data to check, so assume no depeg)
for h in horizons:
    col = f"depeg_future_{h}d"
    df[col] = df[col].fillna(0).astype(int)

df.head()

,Unnamed: 0,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,...,volume,marketCap,circulatingSupply,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d
0,0,UST,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 11:48:07,2020-11-25 17:19:07,1.010488,1.010752,0.993542,...,3758726.64,1.332804e+07,13333749.41,0,0,0,0,0,0,0
1,1,UST,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 03:33:07,2020-11-26 09:47:07,0.999360,1.022774,0.982691,...,16210369.84,1.564638e+07,15639476.02,0,0,0,0,0,0,0
2,2,UST,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 11:33:07,2020-11-27 16:07:07,1.000106,1.003605,0.994698,...,4086972.72,1.743930e+07,17447758.34,0,0,0,0,0,0,0
3,3,UST,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 19:25:11,2020-11-28 16:04:08,0.999476,1.004033,0.995601,...,3721757.38,1.743599e+07,17447757.88,0,0,0,0,0,0,0
4,4,UST,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 16:06:07,2020-11-29 21:04:07,0.999246,1.001828,0.997059,...,856953.09,1.743870e+07,17444966.35,0,0,0,0,0,0,0


In [38]:
# reorder columns (place labels after first column) ---
cols = df.columns.tolist()

first_col = cols[0]
label_cols = ["depeg"] + [f"depeg_future_{h}d" for h in horizons]

# remove label cols first
remaining_cols = [c for c in cols if c not in label_cols]

# rebuild column order
new_cols = [first_col] + label_cols + [c for c in remaining_cols if c != first_col]

df = df[new_cols]

df.head()

,Unnamed: 0,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,symbol,timestamp,...,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,0,0,0,0,0,0,0,0,UST,2020-11-25 23:59:59,...,2020-11-25 23:59:59,2020-11-25 11:48:07,2020-11-25 17:19:07,1.010488,1.010752,0.993542,0.999572,3758726.64,1.332804e+07,13333749.41
1,1,0,0,0,0,0,0,0,UST,2020-11-26 23:59:59,...,2020-11-26 23:59:59,2020-11-26 03:33:07,2020-11-26 09:47:07,0.999360,1.022774,0.982691,1.000441,16210369.84,1.564638e+07,15639476.02
2,2,0,0,0,0,0,0,0,UST,2020-11-27 23:59:59,...,2020-11-27 23:59:59,2020-11-27 11:33:07,2020-11-27 16:07:07,1.000106,1.003605,0.994698,0.999515,4086972.72,1.743930e+07,17447758.34
3,3,0,0,0,0,0,0,0,UST,2020-11-28 23:59:59,...,2020-11-28 23:59:59,2020-11-28 19:25:11,2020-11-28 16:04:08,0.999476,1.004033,0.995601,0.999325,3721757.38,1.743599e+07,17447757.88
4,4,0,0,0,0,0,0,0,UST,2020-11-29 23:59:59,...,2020-11-29 23:59:59,2020-11-29 16:06:07,2020-11-29 21:04:07,0.999246,1.001828,0.997059,0.999641,856953.09,1.743870e+07,17444966.35


---

## Save

In [39]:
df.dtypes

Unnamed: 0                    int64
depeg                         int64
depeg_future_1d               int64
depeg_future_3d               int64
depeg_future_5d               int64
depeg_future_7d               int64
depeg_future_14d              int64
depeg_future_30d              int64
symbol                          str
timestamp            datetime64[us]
timeOpen             datetime64[us]
timeClose            datetime64[us]
timeHigh             datetime64[us]
timeLow              datetime64[us]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [40]:
df.to_parquet(f"clean_data/{stablecoin}_clean.parquet")
